**Лабораторная работа №2**

Решить задачу классификации датасета MNIST используя MLP из scikitlearn и используя CNN (по типу LeNet) c пакетом PyTorch. Сравнить результаты по метрикам, сделать обоснованные выводы.


In [9]:
from sklearn.datasets import fetch_openml
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tensorflow.keras.datasets import mnist

In [2]:
# сеть LeNet
class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5, padding=2)
        self.avg1 = nn.AvgPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.avg2 = nn.AvgPool2d(2, 2)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = torch.tanh(self.conv1(x))
        x = self.avg1(x)
        x = torch.tanh(self.conv2(x))
        x = self.avg2(x)
        x = x.view(x.size(0), -1)  # 16*5*5
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        x = self.fc3(x)
        return x

In [19]:
# Подготовка данных
(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_train = X_train.astype('float32') / 255.0
X_train_mlp = X_train.reshape(-1, 28*28)
X_train_cnn = X_train[:, np.newaxis, :, :]

X_test  = X_test.astype('float32') / 255.0
X_test_mlp  = X_test.reshape(-1, 28*28)
X_test_cnn  = X_test[:, np.newaxis, :, :]

y_train = y_train.astype('int')
y_test  = y_test.astype('int')

print("Размеры:", X_train_mlp.shape, X_test_mlp.shape, X_train_cnn.shape, X_test_cnn.shape)

Размеры: (60000, 784) (10000, 784) (60000, 1, 28, 28) (10000, 1, 28, 28)


In [13]:
# сеть MLP
mlp = MLPClassifier(hidden_layer_sizes=(100,), activation='relu', solver='adam', max_iter=20, batch_size=128, verbose=True, random_state=42)
mlp.fit(X_train_mlp, y_train)

y_pred_mlp = mlp.predict(X_test_mlp)
acc_mlp = accuracy_score(y_test, y_pred_mlp)
print()
print("Точность MLP на тесте:", acc_mlp)
print()
print(classification_report(y_test, y_pred_mlp))

Iteration 1, loss = 0.38642037
Iteration 2, loss = 0.19002429
Iteration 3, loss = 0.14131137
Iteration 4, loss = 0.11172061
Iteration 5, loss = 0.09207674
Iteration 6, loss = 0.07703727
Iteration 7, loss = 0.06566416
Iteration 8, loss = 0.05564364
Iteration 9, loss = 0.04916073
Iteration 10, loss = 0.04229065
Iteration 11, loss = 0.03675807
Iteration 12, loss = 0.03241165
Iteration 13, loss = 0.02798889
Iteration 14, loss = 0.02450835
Iteration 15, loss = 0.02117892
Iteration 16, loss = 0.01882087
Iteration 17, loss = 0.01666361
Iteration 18, loss = 0.01420498
Iteration 19, loss = 0.01266132
Iteration 20, loss = 0.01104794

Точность MLP на тесте: 0.9776

              precision    recall  f1-score   support

           0       0.98      0.99      0.98       980
           1       0.99      0.99      0.99      1135
           2       0.97      0.97      0.97      1032
           3       0.97      0.99      0.98      1010
           4       0.98      0.98      0.98       982
           5

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Преобразование данных в тензоры для CNN
X_train_t = torch.tensor(X_train_cnn).view(-1, 1, 28, 28).to(device)
y_train_t = torch.tensor(y_train).to(device)
X_test_t = torch.tensor(X_test_cnn).view(-1, 1, 28, 28).to(device)
y_test_t = torch.tensor(y_test).to(device)

train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset = TensorDataset(X_test_t, y_test_t)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

cuda


In [15]:
model = LeNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 5
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}")

Epoch 1/5, Loss: 0.3054
Epoch 2/5, Loss: 0.0974
Epoch 3/5, Loss: 0.0648
Epoch 4/5, Loss: 0.0479
Epoch 5/5, Loss: 0.0395


In [18]:
# Оценка на тесте
model.eval()
correct = 0
total = 0
all_preds = []
all_labels = []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

acc_cnn = correct / total
print("Точность CNN:", acc_cnn)
print()
print(classification_report(all_labels, all_preds))

Точность CNN: 0.9858

              precision    recall  f1-score   support

           0       0.98      0.99      0.99       980
           1       0.98      1.00      0.99      1135
           2       0.99      0.99      0.99      1032
           3       0.98      0.99      0.99      1010
           4       0.99      0.98      0.99       982
           5       0.99      0.99      0.99       892
           6       0.98      0.99      0.99       958
           7       0.99      0.98      0.98      1028
           8       1.00      0.98      0.99       974
           9       0.97      0.98      0.98      1009

    accuracy                           0.99     10000
   macro avg       0.99      0.99      0.99     10000
weighted avg       0.99      0.99      0.99     10000



**Вывод:**

Обе сети решили задачу классификации MNIST с высокой точностью (более 97%), однако CNN показала более высокую точность за меньшее количество эпох обучения. Соответственно, использование CNN более эффективно.